In [12]:
import pandas as pd
import numpy as np

df = pd.read_excel(
    "Данные_для_курсовои_Классическое_МО.xlsx"
)
df = df.drop(columns=["Unnamed: 0"])
df = df.fillna(df.median())

In [13]:
targets = ["IC50, mM", "CC50, mM", "SI"]
X = df.drop(columns=targets)

y = (df["CC50, mM"] > df["CC50, mM"].median()).astype(int)

In [14]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42, stratify=y) #гиперпараметр stratify используем чтобы доля классов сохранялась

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score)


lr = LogisticRegression(max_iter=5000, C=1.0) #выбрано оптимальное число итераций обучения и оптимальный штраф за слишком сложную модель

lr.fit(X_train, y_train)

pred = lr.predict(X_test)

print("Logistic Regression")

print("Accuracy:", accuracy_score(y_test, pred))

print("F1:", f1_score(y_test, pred))

print("ROC-AUC:", roc_auc_score(y_test, pred))

Logistic Regression
Accuracy: 0.5024875621890548
F1: 0.0
ROC-AUC: 0.5


Модель логистической регресии пока демонстрирует не самые лучшие значения, модель все еще слаба

Проверим модель случайного леса

In [5]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=700, max_depth=10, min_samples_split=5) #глубина оптимальная, нету недообучения и риска переобучения
                                                                                #min_samples_split делает деревья менее сложными борется с переобучением
rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print("Random Forest")

print("Accuracy:", accuracy_score(y_test, pred))

print("F1:", f1_score(y_test, pred))

print("ROC-AUC:", roc_auc_score(y_test, pred))

Random Forest
Accuracy: 0.7213930348258707
F1: 0.7383177570093458
ROC-AUC: 0.7217326732673268


Данные параметры уже демонстрируют что такая модель очень хорошо понимает наши признаки и действительно находит закономерности, тем более с высокой точностью

Посмотрим на градиент бустинг

In [15]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42, max_depth=10, min_samples_split=5)   #глубина оптимальная, нету недообучения и риска переобучения
                                                                                     #min_samples_split делает деревья менее сложными борется с переобучением
gb.fit(X_train, y_train)

pred = gb.predict(X_test)

print("GradientBoostingClassifier")

print("Accuracy:", accuracy_score(y_test, pred))

print("F1:", f1_score(y_test, pred))

print("ROC-AUC:", roc_auc_score(y_test, pred))


GradientBoostingClassifier
Accuracy: 0.736318407960199
F1: 0.7464114832535885
ROC-AUC: 0.7365346534653465


Для данного признака градиент бустинг показал самые лучшие метрики

Посмотрим как она сработает на 9 наших избранных признаках

In [16]:
selected_features = [
    "VSA_EState8",
    "VSA_EState6",
    "VSA_EState4",
    "MolLogP",
    "qed",
    "FractionCSP3",
    "NumSaturatedHeterocycles",
    "NumSaturatedCarbocycles",
    "NumAliphaticCarbocycles"
]

X_small = df[selected_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_small,
    y,
    test_size=0.2,
    random_state=42
)

In [17]:
gb = GradientBoostingClassifier(random_state=42, max_depth=10, min_samples_split=5)
gb.fit(X_train, y_train)

pred = gb.predict(X_test)

In [19]:
print("GradientBoostingClassifier")

print("Accuracy:", accuracy_score(y_test, pred))

print("F1:", f1_score(y_test, pred))

print("ROC-AUC:", roc_auc_score(y_test, pred))


GradientBoostingClassifier
Accuracy: 0.7810945273631841
F1: 0.7777777777777778
ROC-AUC: 0.7833333333333334


После отбора наиболее информативных дескрипторов качество классификации CC50 заметно улучшилось. Полученный результат показывает, что значительная часть исходных дескрипторов содержала избыточную или шумовую информацию. Отбор признаков, выполненный на этапе EDA, позволил выделить наиболее информативные характеристики химических соединений и повысить качество классификации.